In [ ]:
# 缺失分布分析：按 Group × Phase × Session 交叉计算缺失率
# 对应审稿意见 #6.1，用于补充材料表格

import pandas as pd
import numpy as np
import os

# 设置中文显示
import matplotlib.pyplot as plt
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

import warnings
warnings.filterwarnings('ignore')

In [ ]:
# 读取插值前的原始数据（与 code/EYE/2.数据插值NAN.py 相同输入）
input_file = "眼动数据预处理文件.xlsx"

if not os.path.exists(input_file):
    # 尝试绝对路径
    input_file = r"../数据文件/EYE/眼动数据预处理文件.xlsx"

print(f"读取文件: {input_file}")
df = pd.read_excel(input_file)
print(f"总记录数: {len(df)}")
print(f"列名: {list(df.columns)}")
df.head()

In [ ]:
# 定义眼动指标和缺失判定（0 值视为缺失，与插值脚本一致）
metrics = ['AOI转换次数', '静态注视熵(SGE)', '眼跳注视熵(GTE)']

# 标记缺失
for m in metrics:
    if m in df.columns:
        df[f'{m}_缺失'] = (df[m] == 0) | (df[m].isna())

# 确认列名映射
# 原始数据中的列名可能是中文：'组别', '阶段', '天数'
print("数据列名:", list(df.columns))
print(f"\n组别取值: {df['组别'].unique() if '组别' in df.columns else 'N/A'}")
print(f"阶段取值: {df['阶段'].unique() if '阶段' in df.columns else 'N/A'}")
print(f"天数取值: {sorted(df['天数'].unique()) if '天数' in df.columns else 'N/A'}")

In [ ]:
# 按 Group × Phase × Session 三维交叉计算缺失率
group_col = '组别'
phase_col = '阶段'
session_col = '天数'  # 1-7

# 对每个指标，计算每组的缺失率
all_tables = {}

for metric in metrics:
    missing_col = f'{metric}_缺失'
    if missing_col not in df.columns:
        print(f"跳过 {metric}: 列不存在")
        continue
    
    # 交叉聚合
    agg = df.groupby([group_col, phase_col, session_col]).agg(
        总记录数=(missing_col, 'count'),
        缺失数=(missing_col, 'sum')
    ).reset_index()
    
    agg['缺失率'] = (agg['缺失数'] / agg['总记录数'] * 100).round(2)
    agg['缺失率_显示'] = agg.apply(
        lambda r: f"{r['缺失率']:.1f}% ({int(r['缺失数'])}/{int(r['总记录数'])})", axis=1
    )
    
    # 透视表：行=阶段, 列=(组别, 天数), 值=缺失率
    pivot = agg.pivot_table(
        index=phase_col,
        columns=[group_col, session_col],
        values='缺失率',
        aggfunc='first'
    )
    pivot = pivot.round(1)
    
    all_tables[metric] = {
        'agg': agg,
        'pivot': pivot
    }
    
    print(f"\n{'='*60}")
    print(f"{metric} — 缺失率 (%)  Group×Phase×Session")
    print(f"{'='*60}")
    print(pivot.to_string())
    print()

In [ ]:
# 汇总统计：按 Group 的总体缺失率
print("\n=== 按组别汇总 ===")
for metric in metrics:
    missing_col = f'{metric}_缺失'
    if missing_col not in df.columns:
        continue
    summary = df.groupby(group_col).agg(
        总记录数=(missing_col, 'count'),
        缺失数=(missing_col, 'sum')
    )
    summary['缺失率'] = (summary['缺失数'] / summary['总记录数'] * 100).round(2)
    print(f"\n{metric}:")
    print(summary.to_string())

In [ ]:
# 汇总统计：按 Phase 的总体缺失率
print("\n=== 按阶段汇总 ===")
for metric in metrics:
    missing_col = f'{metric}_缺失'
    if missing_col not in df.columns:
        continue
    summary = df.groupby(phase_col).agg(
        总记录数=(missing_col, 'count'),
        缺失数=(missing_col, 'sum')
    )
    summary['缺失率'] = (summary['缺失数'] / summary['总记录数'] * 100).round(2)
    print(f"\n{metric}:")
    print(summary.to_string())

In [ ]:
# 汇总统计：按 Session 的总体缺失率
print("\n=== 按 Session 汇总 ===")
for metric in metrics:
    missing_col = f'{metric}_缺失'
    if missing_col not in df.columns:
        continue
    summary = df.groupby(session_col).agg(
        总记录数=(missing_col, 'count'),
        缺失数=(missing_col, 'sum')
    )
    summary['缺失率'] = (summary['缺失数'] / summary['总记录数'] * 100).round(2)
    print(f"\n{metric}:")
    print(summary.to_string())

In [ ]:
# 检查缺失是否与组别系统相关（MCAR 检验的思路）
print("\n=== 缺失模式检验：Group × Session 的缺失计数 ===")

for metric in metrics:
    missing_col = f'{metric}_缺失'
    if missing_col not in df.columns:
        continue
    
    # Group × Session 交叉
    cross = df.groupby([group_col, session_col]).agg(
        总记录数=(missing_col, 'count'),
        缺失数=(missing_col, 'sum')
    )
    cross['缺失率'] = (cross['缺失数'] / cross['总记录数'] * 100).round(2)
    
    print(f"\n{metric}:")
    print(cross.to_string())
    
    # 卡方检验：缺失是否与组别相关
    from scipy.stats import chi2_contingency
    alcohol_missing = cross.loc['Alcohol']['缺失数'] if 'Alcohol' in cross.index else 0
    alcohol_total = cross.loc['Alcohol']['总记录数'] if 'Alcohol' in cross.index else 0
    control_missing = cross.loc['Control']['缺失数'] if 'Control' in cross.index else 0
    control_total = cross.loc['Control']['总记录数'] if 'Control' in cross.index else 0
    
    contingency = pd.DataFrame({
        '缺失': [alcohol_missing, control_missing],
        '非缺失': [alcohol_total - alcohol_missing, control_total - control_missing]
    }, index=['Alcohol', 'Control'])
    
    try:
        chi2, p, dof, expected = chi2_contingency(contingency)
        print(f"  χ² = {chi2:.3f}, p = {p:.4f} (Group vs Missing)")
    except:
        print(f"  无法计算卡方检验")

In [ ]:
# 保存结果到 Excel（补充材料表格）
output_file = "缺失分布_Group_Phase_Session.xlsx"

with pd.ExcelWriter(output_file) as writer:
    for metric in metrics:
        if metric in all_tables:
            # 透视表
            sheet_name = metric[:31]  # Excel sheet name limit
            all_tables[metric]['pivot'].to_excel(writer, sheet_name=sheet_name)
            
            # 详细数据
            all_tables[metric]['agg'].to_excel(
                writer, 
                sheet_name=f"{metric[:25]}_详细",
                index=False
            )

print(f"结果已保存至: {output_file}")
print("\n完成！将此文件作为补充材料 Table S? 提交。")